# Notebook 05 — Analyse de cadrage VSS × Immigration

## Objectif
Ce notebook est le cœur de l'analyse. On cherche à **classifier** chaque phrase qui mentionne à la fois les VSS et l'immigration/l'islam en trois catégories :

- **ACCUSATEUR** : l'orateur présente l'immigration comme la *cause* des VSS
- **VICTIME** : l'orateur présente les immigrés/minorités comme *victimes* de violences
- **NEUTRE** : mention administrative, factuelle, ou dénonciation de l'amalgame

## Méthodes (par ordre de sophistication)
1. **V1 — Lexique de cadrage-menace** : score basé sur des listes de mots de causalité/accusation
2. **V2 — Patterns orientés rôle** : regex syntaxiques (sujet-verbe-objet) pour détecter qui est agent vs victime
3. **V3 — Zero-shot NLI** : modèles Transformers (mDeBERTa, CamemBERT) pour classification sémantique
4. **V4 — Classification LLM** : Llama 3.3 / Mistral via le serveur Ollama de l'ENSAE

## Entrées
- `df_vss_propre.pkl` (notebook 02)

## Sorties
- CSV des phrases classifiées avec justification
- Graphiques : barres empilées, score net, évolution temporelle, heatmap par parti

## 1. Imports et chargement

In [ ]:
from config import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import re, os, time
from tqdm import tqdm

df_vss = pd.read_pickle(CHEMIN_DF_VSS_PROPRE)
df_vss['date'] = pd.to_datetime(df_vss['date'])
df_vss = df_vss.dropna(subset=['bloc'])
print(f"✅ {len(df_vss)} prises de parole VSS chargées.")

## 2. Extraction des phrases de co-occurrence

On ne s'intéresse qu'aux phrases qui mentionnent **à la fois** un mot-clé VSS et un mot-clé identitaire/migratoire. On utilise deux modes :
- **Direct** : les deux thématiques dans la même phrase
- **Fenêtre** : un mot-clé VSS dans une phrase, un mot-clé identitaire dans une phrase voisine (±2 phrases)

In [ ]:
def extraire_contextes(df_vss, fenetre_phrases=2):
    """
    Extrait les contextes où VSS et immigration se croisent.

    Returns:
        DataFrame avec colonnes [contexte, bloc, nom_parti, date, mode]
    """
    pat_groupe = re.compile(
        r"(?i)\b(" + "|".join([re.escape(m) for m in MOTS_IDENTITAIRES]) + r")\w*\b"
    )
    pat_vss = re.compile(
        r"(?i)\b(" + "|".join([re.escape(m) for m in MOTS_VSS_VIOLENCE]) + r")\w*\b"
    )

    resultats = []
    for _, row in tqdm(df_vss.iterrows(), total=len(df_vss), desc="Extraction"):
        texte = str(row.get("texte", ""))
        bloc = row.get("bloc")
        if not texte or pd.isna(bloc):
            continue

        phrases = [p.strip() for p in re.split(r"[.!?;]+", texte) if len(p.strip().split()) >= 5]
        n = len(phrases)

        for i, phrase in enumerate(phrases):
            a_vss = bool(pat_vss.search(phrase))
            a_groupe = bool(pat_groupe.search(phrase))

            if a_vss and a_groupe:
                resultats.append({
                    "contexte": phrase, "bloc": bloc,
                    "nom_parti": row.get("nom_parti"), "date": row.get("date"),
                    "mode": "directe"
                })
            elif a_vss:
                debut = max(0, i - fenetre_phrases)
                fin = min(n, i + fenetre_phrases + 1)
                voisines = phrases[debut:i] + phrases[i+1:fin]
                if any(pat_groupe.search(v) for v in voisines):
                    resultats.append({
                        "contexte": " ".join(phrases[debut:fin]),
                        "bloc": bloc, "nom_parti": row.get("nom_parti"),
                        "date": row.get("date"), "mode": "fenetre"
                    })

    df = pd.DataFrame(resultats).drop_duplicates(subset="contexte")
    print(f"\n✅ {len(df)} contextes extraits ({df['mode'].value_counts().to_dict()})")
    print(df["bloc"].value_counts().to_string())
    return df

df_contextes = extraire_contextes(df_vss)

## 3. Méthode V2 — Patterns orientés rôle (regex)

On détecte si le groupe cible (immigrés, musulmans...) est **agent** (sujet d'un verbe d'agression) ou **patient** (victime de violences) dans la phrase. Cette approche est plus fine que le simple comptage de mots-clés car elle tient compte de la structure syntaxique.

In [ ]:
# Patterns où le groupe cible est AGENT (accusateur)
PATTERNS_ACCUSATEUR = [
    r"(immigr\w*|clandestin\w*|étranger\w*|migrant\w*|musulman\w*|arabe\w*)"
    r"[\w\s,]{0,30}"
    r"(viol\w*|agress\w*|maltraitent|commettent|perpètrent|battent|frappent|tuent)",

    r"(violences?|viols?|agressions?|féminicides?)"
    r"[\w\s,]{0,20}(commis|perpétr|fait|exerc)\w*"
    r"[\w\s,]{0,15}(immigr\w*|clandestin\w*|étranger\w*|musulman\w*|arabe\w*)",

    r"(à cause de|en raison de|du fait de|liés? à)"
    r"[\w\s]{0,20}(l.immigration|clandestin|leur culture|leur religion|l.islam)",

    r"(import[ée]\w*|venu[e]? d.ailleurs|étranger[eès]? à notre|apporté\w*)",
    r"(nos femmes|nos filles|nos enfants)[\w\s,]{0,30}(protéger|défendre|menac\w*|danger)",
    r"(invasion|ensauvagement|grand remplacement|submersion|islamisation)",
    r"(leur culture|leur mentalité|leur rapport à la femme|leur vision de la femme)",
]

# Patterns où le groupe cible est VICTIME
PATTERNS_VICTIME = [
    r"(immigr\w*|étranger\w*|migrant\w*|musulman\w*|sans-papier\w*|réfugié\w*)"
    r"[\w\s,]{0,30}(victimes?|subiss\w*|agress[ée]\w*|discrimin\w*|exposés?)",

    r"(protéger|défendre|accompagner|soutenir|aider)"
    r"[\w\s]{0,20}(immigr\w*|migrant\w*|sans-papier\w*|réfugiées?)",

    r"(doublement|vulnérable|précaire|plus exposée|davantage exposée)",
    r"(bouc[s]? émissaire[s]?|stigmatis\w*|diabolis\w*|montré[s]? du doigt)",
    r"(amalgam\w*|instrumentalis\w*|récupération)",
]

def detecter_role(phrase):
    phrase_lower = phrase.lower()
    n_acc = sum(1 for p in PATTERNS_ACCUSATEUR if re.search(p, phrase_lower, re.I | re.DOTALL))
    n_vic = sum(1 for p in PATTERNS_VICTIME if re.search(p, phrase_lower, re.I | re.DOTALL))

    score_net = n_acc - n_vic
    if n_acc == 0 and n_vic == 0:
        cadrage = "NEUTRE"
    elif n_acc > n_vic:
        cadrage = "ACCUSATEUR"
    elif n_vic > n_acc:
        cadrage = "VICTIME"
    else:
        cadrage = "NEUTRE"
    return {"n_accusateur": n_acc, "n_victime": n_vic, "score_net": score_net, "cadrage": cadrage}

# Application
scores = df_contextes["contexte"].apply(detecter_role)
df_scored = pd.concat([df_contextes.reset_index(drop=True), pd.DataFrame(scores.tolist())], axis=1)

print(f"\n📊 Distribution des cadrages (regex) :")
print(df_scored['cadrage'].value_counts().to_string())

### Visualisation V2 — Barres empilées et score net

In [ ]:
def tracer_resultats(df, col_cadrage="cadrage", titre_suffixe=""):
    blocs_presents = [b for b in ORDRE_BLOCS if b in df["bloc"].unique()]
    counts = df.groupby(["bloc", col_cadrage]).size().unstack(fill_value=0).reindex(blocs_presents)
    pct = counts.div(counts.sum(axis=1), axis=0) * 100
    for col in ["ACCUSATEUR", "NEUTRE", "VICTIME"]:
        if col not in pct.columns:
            pct[col] = 0

    coul = {"ACCUSATEUR": "#C0392B", "NEUTRE": "#95A5A6", "VICTIME": "#2980B9"}

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    # Barres empilées
    bottom = np.zeros(len(pct))
    for cat in ["ACCUSATEUR", "NEUTRE", "VICTIME"]:
        vals = pct[cat].values
        axes[0].bar(pct.index, vals, bottom=bottom, color=coul[cat], label=cat, edgecolor="white", lw=0.5)
        for i, (v, b) in enumerate(zip(vals, bottom)):
            if v > 4:
                axes[0].text(i, b + v/2, f"{v:.0f}%", ha="center", va="center", fontsize=9, color="white", fontweight="bold")
        bottom += vals
    axes[0].set_ylim(0, 100)
    axes[0].set_ylabel("Part des contextes (%)")
    axes[0].set_title(f"Cadrage de l'immigration dans les VSS{titre_suffixe}")
    axes[0].legend(bbox_to_anchor=(1.01, 1), loc="upper left")
    axes[0].tick_params(axis='x', rotation=20)

    # Score net
    stats = df.groupby("bloc")["score_net"].agg(["mean", "sem", "count"]).reset_index()
    stats = stats[stats["bloc"].isin(ORDRE_BLOCS)]
    stats["bloc"] = pd.Categorical(stats["bloc"], categories=ORDRE_BLOCS, ordered=True)
    stats = stats.sort_values("bloc")
    couleurs = [COULEURS_BLOCS.get(b, "gray") for b in stats["bloc"]]

    axes[1].barh(stats["bloc"], stats["mean"], xerr=stats["sem"], color=couleurs, edgecolor="white", capsize=4)
    axes[1].axvline(0, color="black", lw=1.5)
    axes[1].set_title(f"Score net (accusateur − victime){titre_suffixe}")
    axes[1].set_xlabel("← protecteur | accusateur →")
    axes[1].grid(axis="x", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

tracer_resultats(df_scored, titre_suffixe="\n(Méthode regex)")

## 4. Méthode V4 — Classification par LLM (Llama/Mistral via Ollama)

On envoie chaque contexte à un LLM avec un prompt structuré qui lui demande de classifier la phrase en ACCUSATEUR / VICTIME / NEUTRE, avec une justification. Le prompt inclut des "pièges à éviter" pour améliorer la précision.

> **⏱ Temps estimé** : ~1-3h selon le nombre de contextes et la charge du serveur.
> **⚠️ Nécessite** le serveur Ollama de l'ENSAE.

In [ ]:
import requests, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

MODELE_CHOISI = "llama3.3"
output_dir_llm = os.path.join(DOSSIER_ANALYSES, "cadrage_llm")
os.makedirs(output_dir_llm, exist_ok=True)
fichier_csv_llm = os.path.join(output_dir_llm, "phrases_classifiees_llm.csv")

PROMPT_SYSTEME = (
    "Tu es un chercheur en sociologie politique expert en analyse de discours parlementaire français. "
    "Analyse le contexte fourni et détermine le cadrage utilisé par l'orateur.\n\n"
    "PIEGES A EVITER :\n"
    "- DENONCIATION : Si l'orateur critique l'amalgame immigration-VSS => NEUTRE\n"
    "- VOIX PASSIVE : Si un étranger 'subit' ou 'est agressé' => VICTIME\n"
    "- ADMINISTRATIF : Conditions d'éloignement, droit d'asile => NEUTRE\n\n"
    "CATEGORIES :\n"
    "  ACCUSATEUR : l'orateur affirme que l'immigration CAUSE les VSS\n"
    "  VICTIME : l'orateur présente les immigrés comme VICTIMES\n"
    "  NEUTRE : réfutation, fait administratif, sans lien causal\n\n"
    "REPONSE en 2 lignes :\n"
    "ANALYSE : [justification courte]\n"
    "CATEGORIE : [ACCUSATEUR ou VICTIME ou NEUTRE]"
)

def appeler_llm(prompt_user, max_retries=6):
    payload = {
        "model": MODELE_CHOISI,
        "messages": [
            {"role": "system", "content": PROMPT_SYSTEME},
            {"role": "user", "content": prompt_user},
        ],
        "stream": False,
        "options": {"temperature": 0.0},
    }
    for attempt in range(max_retries):
        try:
            r = requests.post(URL_OLLAMA, json=payload, timeout=90, verify=False)
            r.raise_for_status()
            return r.json()["message"]["content"]
        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 504:
                time.sleep(30)
            elif e.response.status_code == 404:
                print(f"\n❌ Modèle '{MODELE_CHOISI}' introuvable.")
                return ""
            else:
                time.sleep(10)
        except requests.exceptions.RequestException:
            time.sleep(30)
    return ""

print(f"✅ Configuration LLM prête (modèle : {MODELE_CHOISI}).")
print(f"   {len(df_contextes)} contextes à classifier.")

In [ ]:
# ==========================================================================
# CACHE : Reprise depuis le CSV existant
# ==========================================================================

deja_faites = set()
if os.path.exists(fichier_csv_llm):
    df_existant = pd.read_csv(fichier_csv_llm)
    deja_faites = set(df_existant["contexte"].tolist())
    print(f"⏩ {len(deja_faites)} contextes déjà classifiés. Reprise en cours...")
else:
    colonnes = list(df_contextes.columns) + ["cadrage_llm", "score_net_llm", "justification", "reponse_brute"]
    pd.DataFrame(columns=colonnes).to_csv(fichier_csv_llm, index=False, encoding="utf-8-sig")

df_restant = df_contextes[~df_contextes["contexte"].isin(deja_faites)]
print(f"   {len(df_restant)} contextes restants à classifier.")

for _, row in tqdm(df_restant.iterrows(), total=len(df_restant), desc="Classification LLM"):
    prompt = (
        f"Parti de l'orateur : '{row['nom_parti']}'\n"
        f"Contexte :\n\"{row['contexte']}\"\n\n"
        "Fournis ton ANALYSE puis ta CATEGORIE."
    )

    reponse = appeler_llm(prompt)
    if not reponse:
        print("\n❌ Arrêt suite à une erreur critique.")
        break

    cadrage, score_net, justification = "NEUTRE", 0, reponse
    match = re.search(r"CATEGORIE\s*:\s*(ACCUSATEUR|VICTIME|NEUTRE)", reponse, re.I)
    if match:
        cadrage = match.group(1).upper()
        score_net = {"ACCUSATEUR": 1, "VICTIME": -1}.get(cadrage, 0)
    match_ana = re.search(r"ANALYSE\s*:\s*(.*?)(?=\nCATEGORIE|$)", reponse, re.I | re.DOTALL)
    if match_ana:
        justification = match_ana.group(1).strip()

    ligne = row.to_dict()
    ligne.update({"cadrage_llm": cadrage, "score_net_llm": score_net,
                  "justification": justification, "reponse_brute": reponse})
    pd.DataFrame([ligne]).to_csv(fichier_csv_llm, mode="a", header=False, index=False, encoding="utf-8-sig")
    time.sleep(0.3)

# Chargement du résultat complet
df_llm = pd.read_csv(fichier_csv_llm)
df_llm["score_net"] = df_llm["cadrage_llm"].map({"ACCUSATEUR": 1, "VICTIME": -1, "NEUTRE": 0}).fillna(0)
print(f"\n✅ Classification terminée : {len(df_llm)} contextes.")
print(df_llm["cadrage_llm"].value_counts().to_string())

### Visualisation des résultats LLM

In [ ]:
tracer_resultats(df_llm, col_cadrage="cadrage_llm", titre_suffixe=f"\n(Classification {MODELE_CHOISI.upper()})")

### Évolution temporelle du cadrage

In [ ]:
d = df_llm.copy()
d["date"] = pd.to_datetime(d["date"], errors="coerce")
d["annee"] = d["date"].dt.year

evol = d.groupby(["annee", "bloc"])["score_net"].mean().reset_index()
evol = evol[evol["bloc"].isin(ORDRE_BLOCS)]

plt.figure(figsize=(13, 7))
for bloc in ORDRE_BLOCS:
    sub = evol[evol["bloc"] == bloc].sort_values("annee")
    if sub.empty:
        continue
    plt.plot(sub["annee"], sub["score_net"], marker="o", label=bloc,
             color=COULEURS_BLOCS.get(bloc, "gray"), lw=2.5, markersize=7)

plt.axhline(0, color="black", lw=1, ls="--", alpha=0.5)
plt.title(f"Évolution du cadrage 'immigration' dans les VSS (2017–2025)\nScore net : + accusateur | − protecteur")
plt.xlabel("Année")
plt.ylabel("Score net moyen")
plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.grid(ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

### Heatmap par parti et par année

In [ ]:
d = df_llm.copy()
d["date"] = pd.to_datetime(d["date"], errors="coerce")
d["annee"] = d["date"].dt.year.astype("Int64")

top_partis = d["nom_parti"].value_counts().head(12).index.tolist()
d = d[d["nom_parti"].isin(top_partis)]

pivot = d.groupby(["nom_parti", "annee"])["score_net"].mean().unstack(fill_value=np.nan)

plt.figure(figsize=(14, 8))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, linewidths=0.4, linecolor="white",
            cbar_kws={"label": "Score net (+ accusateur, − protecteur)"})
plt.title(f"Score de cadrage par parti et par année ({MODELE_CHOISI.upper()})")
plt.xlabel("Année")
plt.ylabel("Parti")
plt.tight_layout()
plt.show()

### Validation qualitative

In [ ]:
print("=" * 80)
print("VALIDATION QUALITATIVE — exemples par bloc")
print("=" * 80)

for bloc in ORDRE_BLOCS:
    sub = df_llm[df_llm["bloc"] == bloc]
    if sub.empty:
        continue
    print(f"\n{'─'*65}")
    print(f"  {bloc.upper()} — {len(sub)} contextes, score net moyen : {sub['score_net'].mean():.3f}")
    print(f"  {sub['cadrage_llm'].value_counts().to_dict()}")

    for cadrage in ["ACCUSATEUR", "VICTIME"]:
        sous = sub[sub["cadrage_llm"] == cadrage].head(2)
        if sous.empty:
            continue
        print(f"\n  [{cadrage}]")
        for _, row in sous.iterrows():
            txt = str(row["contexte"])[:220]
            just = str(row.get("justification", ""))[:120]
            print(f"    [{row['nom_parti']}] \"{txt}...\"")
            print(f"    → {just}")